# CE541E08 — Unit 4 · Day 32 — groupby and resample

| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 4 — The Pandas Library |
| **Session** | Day 32 of 45 |
| **CO** | CO4 |
| **Topics** | groupby · agg · resample ME/YE · decade comparison |

---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge at the end.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 32"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Aggregating Data Over Time

`groupby` and `resample` are the two most important Pandas tools for temporal aggregation in hydrology:

- **`groupby('Year')`** — splits data by a column value and applies a function to each group
- **`resample('ME')`** — groups a DatetimeIndex by time period (month, year, season) and aggregates

The key difference: `groupby` works on any column; `resample` requires a DatetimeIndex.

---
## Code Block 1 — groupby: Annual Statistics

### What this code does

We compute annual mean, peak, low, and total flow from a 10-year monthly DataFrame using `groupby('Year').agg(...)`.

### Why each step is taken

**`groupby('Year')`:**
Splits the 120-row DataFrame (10 years × 12 months) into 10 groups — one per year. All subsequent operations apply independently to each group.

**`.agg(['mean','max','min','sum'])`:**
`agg` applies multiple functions at once. For each year group, it computes the mean monthly flow, peak monthly flow, minimum monthly flow, and annual total. Without `agg`, you would need four separate `groupby` calls.

**`.columns = ['Mean','Peak','Low','Annual_sum']`:**
`agg` creates multi-level column names by default. We replace them with clean single-level names.

**`.idxmax()` and `.idxmin()`:**
Returns the Year label with the highest and lowest annual sum — identifying the wettest and driest years in the record.

### Algorithm

```
1. Build 10-year monthly DataFrame (120 rows)

2. df.groupby('Year')['Flow_m3s'].agg(['mean','max','min','sum'])
   → (10,4) DataFrame — one row per year

3. Rename columns to Mean, Peak, Low, Annual_sum

4. annual['Annual_sum'].idxmax() → wettest year label
   annual['Annual_sum'].idxmin() → driest year label
```

### Expected output

```
      Mean    Peak   Low  Annual_sum
Year
2015   ...     ...   ...       ...
...
2024   ...     ...   ...       ...
Wettest year: 20??
Driest year : 20??
```

In [ ]:
import pandas as pd, numpy as np

np.random.seed(0)
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
base   = [45,38,28,22,35,234,456,389,198,89,62,50]
rows   = []
for yr in range(2015,2025):
    for i,m in enumerate(months):
        rows.append({'Year':yr,'Month':m,'Month_num':i+1,
                     'Flow_m3s':round(base[i]+np.random.normal(0,base[i]*0.2),1)})
df = pd.DataFrame(rows)

# groupby('Year') splits the 120 rows into 10 groups
# .agg([...]) applies multiple functions to each group simultaneously
annual = df.groupby('Year')['Flow_m3s'].agg(['mean','max','min','sum'])
annual.columns = ['Mean','Peak','Low','Annual_sum']

print(annual.round(1))
print(f"Wettest year: {annual['Annual_sum'].idxmax()}")
print(f"Driest year : {annual['Annual_sum'].idxmin()}")

### 🔁 Try this

Compute the **annual standard deviation** by adding `'std'` to the agg list.

Which year had the most variable monthly flow? Is that the same as the wettest year?

---
## Code Block 2 — groupby: Monthly Climatology and Season

### What this code does

We compute the 10-year monthly climatology (mean, std, max for each calendar month) and classify months into seasons using `apply(lambda)`.

### Why each step is taken

**`groupby('Month_num')`:**
Groups by month number (1=Jan, ..., 12=Dec) across all 10 years. For month 6 (June), this collects all 10 June values. `.agg(['mean','std','max'])` then gives the climatological statistics for each month.

**`clim.index = months`:**
Replaces the integer index (1-12) with month name labels — making the output more readable.

**`apply(lambda m:'Monsoon' if 6<=m<=9 else 'Non-monsoon')`:**
Applies a lambda function to every value in `Month_num`. `apply` is the bridge between a Pandas Series and a Python function. This classifies each row into a season based on its month number.

### Algorithm

```
1. df.groupby('Month_num')['Flow_m3s'].agg(['mean','std','max'])
   → one row per month, computed across all 10 years

2. clim.index = months — replace 1-12 with month names

3. df['Season'] = df['Month_num'].apply(lambda m: ...)
   → new column: 'Monsoon' or 'Non-monsoon' for each row

4. df.groupby('Season')['Flow_m3s'].mean()
   → mean flow per season
```

### Expected output

```
Monthly climatology (10-year):
     mean   std    max
Jan   ...   ...    ...
...
Dec   ...   ...    ...

Seasonal means:
Season
Monsoon        ...
Non-monsoon    ...
```

In [ ]:
import pandas as pd, numpy as np

np.random.seed(0)
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
base   = [45,38,28,22,35,234,456,389,198,89,62,50]
rows   = []
for yr in range(2015,2025):
    for i,m in enumerate(months):
        rows.append({'Year':yr,'Month':m,'Month_num':i+1,
                     'Flow_m3s':round(base[i]+np.random.normal(0,base[i]*0.2),1)})
df = pd.DataFrame(rows)

# Group by month number — collects all 10 years of each calendar month
clim = df.groupby('Month_num')['Flow_m3s'].agg(['mean','std','max'])
clim.index = months   # replace 1-12 with month name labels

print("Monthly climatology (10-year):"); print(clim.round(1))

# apply(lambda): applies a function to each value in a column
# Monsoon = months 6,7,8,9; everything else = Non-monsoon
df['Season'] = df['Month_num'].apply(
    lambda m: 'Monsoon' if 6 <= m <= 9 else 'Non-monsoon'
)

print()
print("Seasonal means:")
print(df.groupby('Season')['Flow_m3s'].mean().round(1))

### 🔁 Try this

Add a third season: **Post-monsoon** for months 10 and 11.

Modify the lambda: `'Monsoon' if 6<=m<=9 else ('Post-monsoon' if m in [10,11] else 'Dry')`

Compute the mean for all three seasons.

---
## Code Block 3 — resample: Monthly and Annual Aggregation

### What this code does

We create a 3-year daily DataFrame with a DatetimeIndex and use `.resample()` to convert it to monthly and annual summaries automatically.

### Why each step is taken

**`resample('ME')`:**
`ME` = Month End. Groups all daily records within each calendar month and applies the aggregation. The result has one row per month with the month-end date as the index. `'YE'` = Year End gives one row per year.

**Why resample instead of groupby:**
`resample` understands time — it knows that February has fewer days than July, and it handles the boundary between years automatically. `groupby('Month_num')` would mix all years together; `resample` keeps each month-year combination separate.

### Algorithm

```
1. 3-year daily DataFrame with DatetimeIndex

2. df.resample('ME').agg({'Flow_m3s':['mean','max','min']})
   → one row per month: mean, peak, low flow
   → 36 rows (3 years × 12 months)

3. df.resample('YE').mean()
   → one row per year: annual mean flow
```

### Expected output

```
Monthly resampled (first 6):
             Mean   Peak    Low
Date
2022-01-31   ...    ...    ...
...
2022-06-30   ...    ...    ...

Annual mean:
            Flow_m3s
Date
2022-12-31     ...
...
```

In [ ]:
import pandas as pd, numpy as np

np.random.seed(5)
dates = pd.date_range('2022-01-01','2024-12-31',freq='D')
base  = np.where((dates.month>=6)&(dates.month<=9), 300, 60)
flow  = np.round(np.maximum(base + np.random.normal(0,base*0.25,len(dates)),5), 1)
df    = pd.DataFrame({'Flow_m3s':flow}, index=dates)

# resample('ME') = group by calendar month; requires DatetimeIndex
# agg applies multiple functions: mean, max, min for each month
monthly = df.resample('ME').agg({'Flow_m3s':['mean','max','min']})
monthly.columns = ['Mean','Peak','Low']

print("Monthly resampled (first 6):"); print(monthly.head(6).round(1))

# resample('YE') = group by calendar year
annual = df.resample('YE').mean()
print()
print("Annual mean:"); print(annual.round(1))

### 🔁 Try this

Use `resample('QE')` to compute quarterly mean flow (Q = quarter end).

- How many rows does the result have for 3 years?
- Which quarter consistently has the highest mean flow?

---
## Code Block 4 — Decade Comparison

### What this code does

We create a 25-year DataFrame and compare the first decade vs the second by computing decade-level means and trend direction.

### Why each step is taken

**`(df['Year']//10)*10`:**
Integer division by 10 then multiply by 10 rounds each year down to the decade boundary: 2000→2000, 2009→2000, 2010→2010, etc. This creates a Decade column for groupby.

**Comparing `iloc[:10]` vs `iloc[10:]`:**
Selects the first 10 years and last 15 years of the annual summary DataFrame by row position. Computing the mean of each slice gives the decade averages.

### Algorithm

```
1. Build 25-year annual DataFrame with simulated declining trend

2. df['Decade'] = (df['Year']//10)*10 → decade label per row

3. df.groupby('Decade')[['Flow','Rain']].mean()
   → mean per decade

4. early  = annual.iloc[:10]['Flow'].mean()  → first 5 years
   recent = annual.iloc[10:]['Flow'].mean()  → last 5 years
   Compare and print trend direction
```

### Expected output

```
Decade averages:
      Flow    Rain
2000  444.4   941.6
2010  437.1   929.5
2020  431.7   920.9
Early (2000-04) : 447.3 m3/s
Recent (2020-24): 430.0 m3/s
Change          : -3.9%
```

In [ ]:
import pandas as pd, numpy as np

np.random.seed(7)
years       = list(range(2000, 2025))
annual_flow = [450-i*0.5+np.random.normal(0,30) for i,yr in enumerate(years)]
annual_rain = [950-i*1.2+np.random.normal(0,80) for i,yr in enumerate(years)]

df = pd.DataFrame({'Year':years,'Flow':annual_flow,'Rain':annual_rain})

# Decade column: (year//10)*10 rounds to decade start
df['Decade'] = (df['Year']//10)*10

decade = df.groupby('Decade')[['Flow','Rain']].mean()
print("Decade averages:"); print(decade.round(1))

# Compare first and most recent 5-year period
early  = df[df['Year']<=2004]['Flow'].mean()
recent = df[df['Year']>=2020]['Flow'].mean()
print(f"Early (2000-04) : {early:.1f} m3/s")
print(f"Recent (2020-24): {recent:.1f} m3/s")
print(f"Change          : {(recent-early)/early*100:+.1f}%")

### 🔁 Try this

Change the trend from `-0.5` per year to `-2.0` — a steeper decline.

- Does the trend detection (Increasing/Decreasing) still work?
- What is the new percentage change?

---
## Session Summary — groupby and resample

| Method | Syntax | What it gives |
|---|---|---|
| Annual stats | `df.groupby('Year')['col'].agg([...])` | One row per year |
| Monthly climatology | `df.groupby('Month_num')['col'].agg(...)` | One row per calendar month |
| Season label | `df['col'].apply(lambda m:...)` | New column with season name |
| Monthly resample | `df.resample('ME').agg(...)` | One row per month (DatetimeIndex) |
| Annual resample | `df.resample('YE').mean()` | One row per year (DatetimeIndex) |
| Decade grouping | `(df['Year']//10)*10` | Decade label column |
| Wettest year | `annual['sum'].idxmax()` | Year label with max total |

---
## Day 32 Assignment

Using the 10-year monthly flow DataFrame from Code Block 1:

1. Find the peak monsoon year (highest mean flow in Jun-Sep) using `groupby` and boolean filter
2. Compare the first 5-year period (2015-2019) vs second 5-year period (2020-2024) mean flow
3. State whether the trend is Increasing or Decreasing

### ▶ Assignment cell

In [ ]:
import pandas as pd, numpy as np

np.random.seed(0)
months=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
base=[45,38,28,22,35,234,456,389,198,89,62,50]
rows=[]
for yr in range(2015,2025):
    for i,m in enumerate(months):
        rows.append({'Year':yr,'Month':m,'Month_num':i+1,
                     'Flow_m3s':round(base[i]+np.random.normal(0,base[i]*0.2),1)})
df=pd.DataFrame(rows)

monsoon_df   = df[df['Month_num'].between(6,9)]
monsoon_ann  = monsoon_df.groupby('Year')['Flow_m3s'].mean()
peak_year    = monsoon_ann.idxmax()

decade1 = df[df['Year']<=2019]['Flow_m3s'].mean()
decade2 = df[df['Year']>=2020]['Flow_m3s'].mean()
trend   = "Increasing" if decade2>decade1 else "Decreasing"

print(f"Peak monsoon year: {peak_year}")
print(f"2015-19 mean: {decade1:.1f} m3/s  |  2020-24 mean: {decade2:.1f} m3/s")
print(f"Trend: {trend}")

---
- [ ] Run all cells — verify outputs match expected outputs above
- [ ] Complete the assignment cell (replace `???` placeholders)
- [ ] Upload to GitHub: `Unit4_Pandas/CE541E08_U4_Day32.ipynb`
- [ ] Commit message: `Day 32 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*